# Player $\theta$ evaluation

We evaluate the per-player tilt $\hat\theta$ stored in `artifacts/player_thetas.json` by asking the only honest question: **does adding $\hat\theta$ on top of the population baseline improve action prediction on held-out decisions made by that player?**

The motivation is direct. The model is

$$P_\theta(a \mid h, s) \propto P_{\mathrm{base}}(a \mid h, s)\cdot \exp\!\big(\theta^\top u(a)\big),$$

and $\theta = (0,0,0)$ recovers $P_{\mathrm{base}}$ exactly. So the natural counterfactual is: for each held-out decision the player actually made, what would we have predicted with $\theta = 0$ (population baseline) and what do we predict with $\theta = \hat\theta$ (player-tilted)?

Held-out scope: we evaluate on the **online split** — the 20% of sessions never seen by the global-prior fit *and* never seen by the EM that fit $\theta$. This is the only data that is truly held out from $\hat\theta$. We additionally report on the EM split as the in-sample reference.

Metrics — five, each with a clear motivation:

1. **NLL / cross-entropy**: $-\log P_\theta(a^\star \mid h, s)$. The training-equivalent loss; if $\hat\theta$ is doing useful work it should beat $\theta=0$ in expectation on held-out rows.
2. **Brier score**: less sensitive to overconfidence than NLL; complements (1).
3. **Top-1 accuracy** and **argmax flip rate**: how often does the modal action change between the two predictors? Flipping is a strong claim — $\hat\theta$ is not just sharpening probabilities, it is changing decisions.
4. **Mean realised-action shift** $|\Delta P(a^\star)|$: per-row magnitude of the probability change on the action that actually happened. Distinguishes "the tilt is small but well-aimed" from "the tilt is large but undirected."
5. **Held-out gradient norm** $\|\nabla_\theta\mathcal{Q}\|$ at $\hat\theta$: if the M-step gradient on held-out data is small, $\hat\theta$ is at a local optimum of the held-out objective too — i.e. it generalises. If it's large, EM has overfit the EM split.


## Setup

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from typing import List, Tuple

import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path("/home/stat221/jinyangli/bayesian-poker")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pipeline_common import (
    flatten_hands,
    read_session_names_file,
    split_session_names,
    hole_cards_to_hand_class,
    preflop_decisions_for_hand,
)
from utils.prior.preflop import (
    PreflopPrior,
    canonical_preflop_action,
    FOLD as PRE_FOLD, CHECK_CALL as PRE_CALL, RAISE as PRE_RAISE,
)
from utils.prior.postflop import (
    PostflopPrior,
    PostflopFeatures,
    FOLD as POST_FOLD, CALL as POST_CALL, RAISE as POST_RAISE,
)
from utils.postflop_runner_bridge import collect_postflop_observations_known_hole_cards
from utils.eval.brier import multiclass_brier

### Load $\beta$, $\theta$, and the split definitions

In [ ]:
priors = json.loads((REPO_ROOT / "artifacts" / "global_priors.json").read_text())
BETA_PRE      = np.asarray(priors["preflop"]["beta_preflop"],  dtype=float)
BETA_FACING   = np.asarray(priors["postflop"]["beta_facing"],  dtype=float)
BETA_NO_BET   = np.asarray(priors["postflop"]["beta_no_bet"],  dtype=float)

thetas_payload = json.loads((REPO_ROOT / "artifacts" / "player_thetas.json").read_text())
PLAYERS = list(thetas_payload["players"].keys())
THETA_PRE  = {p: tuple(thetas_payload["players"][p]["theta_pre"])  for p in PLAYERS}
THETA_POST = {p: tuple(thetas_payload["players"][p]["theta_post"]) for p in PLAYERS}

for p in PLAYERS:
    print(f"{p}: theta_pre={tuple(round(x,4) for x in THETA_PRE[p])}"
          f"  theta_post={tuple(round(x,4) for x in THETA_POST[p])}")

session_names = read_session_names_file(REPO_ROOT / "sessions.txt")
train_s, em_s, online_s = split_session_names(
    session_names, train_frac=0.5, em_frac=0.3, online_frac=0.2, seed=42,
)
print("em sessions:", em_s, " online sessions:", online_s)

em_refs     = flatten_hands([REPO_ROOT / "pluribus" / s for s in em_s])
online_refs = flatten_hands([REPO_ROOT / "pluribus" / s for s in online_s])
print(f"em hands: {len(em_refs)}   online hands: {len(online_refs)}")

## Collect each player's held-out decisions

We need the rows where this specific player acted *and* their hole cards are recorded — without hole cards we cannot evaluate the prior conditioned on the realised hand.

In [ ]:
def collect_player_preflop(refs, player):
    """List of (hand_class, state_key, action_label_in_{0,1,2}) for this player."""
    rows = []
    for ref in refs:
        if player not in ref.hand.player_names:
            continue
        hc = hole_cards_to_hand_class(ref.hand.hole_cards.get(player, "") or "")
        if hc is None:
            continue
        for dec in preflop_decisions_for_hand(ref.hand, player, ref.global_index):
            rows.append((hc, dec.state_key, canonical_preflop_action(dec.action_bucket)))
    return rows

def collect_player_postflop(refs, player):
    """Two lists: facing-bet and no-bet (PostflopFeatures, action) rows."""
    facing, no_bet = [], []
    for ref in refs:
        if player not in ref.hand.player_names:
            continue
        obs = collect_postflop_observations_known_hole_cards(ref.hand, player, ref.global_index)
        if obs is None:
            continue
        for feat, action in obs.decisions:
            if feat.facing_bet:
                facing.append((feat, int(action)))
            else:
                if int(action) == POST_FOLD:
                    continue   # drop non-facing folds (matches training-time filter)
                no_bet.append((feat, int(action)))
    return facing, no_bet

DATA_PRE  = {(p, sp): collect_player_preflop(refs, p)
             for p in PLAYERS
             for sp, refs in (("em", em_refs), ("online", online_refs))}
DATA_POST = {(p, sp): collect_player_postflop(refs, p)
             for p in PLAYERS
             for sp, refs in (("em", em_refs), ("online", online_refs))}

print(f"{'player':<10} {'split':<7} {'preflop':>8} {'pf_face':>8} {'pf_no_bet':>10}")
for p in PLAYERS:
    for sp in ("em", "online"):
        f, n = DATA_POST[(p, sp)]
        print(f"{p:<10} {sp:<7} {len(DATA_PRE[(p, sp)]):>8} {len(f):>8} {len(n):>10}")

## Compute action distributions under $\theta=0$ and $\theta=\hat\theta$

`PreflopPrior` and `PostflopPrior` already implement the tilted softmax exactly the way EM does, so we just instantiate two priors per player (one with $\theta=0$, one with $\hat\theta$) and ask for `action_probs`. We collect $(P_0, P_\theta, y)$ matrices per `(player, split, head)`.

In [ ]:
def _to_array_3(probs_dict, idx_a, idx_b, idx_c):
    return np.array([probs_dict[idx_a], probs_dict[idx_b], probs_dict[idx_c]], dtype=float)

def predict_preflop(rows, prior):
    if not rows:
        return np.zeros((0, 3)), np.zeros((0,), dtype=int)
    P = np.zeros((len(rows), 3))
    y = np.zeros(len(rows), dtype=int)
    for i, (hc, sk, a) in enumerate(rows):
        probs = prior.action_probs(hc, sk)
        P[i] = _to_array_3(probs, PRE_FOLD, PRE_CALL, PRE_RAISE)
        y[i] = int(a)
    return P, y

def predict_postflop_facing(rows, prior):
    if not rows:
        return np.zeros((0, 3)), np.zeros((0,), dtype=int)
    P = np.zeros((len(rows), 3))
    y = np.zeros(len(rows), dtype=int)
    for i, (feat, a) in enumerate(rows):
        probs = prior.action_probs(feat)
        P[i] = _to_array_3(probs, POST_FOLD, POST_CALL, POST_RAISE)
        y[i] = int(a)
    return P, y

def predict_postflop_no_bet(rows, prior):
    if not rows:
        return np.zeros((0, 2)), np.zeros((0,), dtype=int)
    P = np.zeros((len(rows), 2))
    y = np.zeros(len(rows), dtype=int)
    for i, (feat, a) in enumerate(rows):
        probs = prior.action_probs(feat)
        # legal actions are CALL=1, RAISE=2; remap to local 0,1
        P[i, 0] = probs[POST_CALL]
        P[i, 1] = probs[POST_RAISE]
        y[i] = 0 if int(a) == POST_CALL else 1
    return P, y

def make_priors(player):
    pre0 = PreflopPrior(theta_pre=(0.0, 0.0, 0.0), beta_preflop=BETA_PRE)
    preH = PreflopPrior(theta_pre=THETA_PRE[player], beta_preflop=BETA_PRE)
    post0 = PostflopPrior(theta_post=(0.0, 0.0, 0.0), beta_facing=BETA_FACING, beta_no_bet=BETA_NO_BET)
    postH = PostflopPrior(theta_post=THETA_POST[player], beta_facing=BETA_FACING, beta_no_bet=BETA_NO_BET)
    return pre0, preH, post0, postH

PRED = {}  # PRED[(player, split, head)] = (P0, P_theta, y)
for player in PLAYERS:
    pre0, preH, post0, postH = make_priors(player)
    for sp in ("em", "online"):
        rows_pre = DATA_PRE[(player, sp)]
        P0, y    = predict_preflop(rows_pre, pre0)
        Pt, _    = predict_preflop(rows_pre, preH)
        PRED[(player, sp, "preflop")] = (P0, Pt, y)

        rows_f, rows_n = DATA_POST[(player, sp)]
        P0, y    = predict_postflop_facing(rows_f, post0)
        Pt, _    = predict_postflop_facing(rows_f, postH)
        PRED[(player, sp, "facing")] = (P0, Pt, y)

        P0, y    = predict_postflop_no_bet(rows_n, post0)
        Pt, _    = predict_postflop_no_bet(rows_n, postH)
        PRED[(player, sp, "no_bet")] = (P0, Pt, y)

print("computed predictions for", len(PRED), "buckets")

## Metric 1 — NLL / cross-entropy

$\mathrm{NLL} = -\frac{1}{N}\sum_i \log P(a^\star_i\mid \phi_i)$. We report $\theta=0$, $\theta=\hat\theta$, and the absolute drop $\Delta = \mathrm{NLL}_0 - \mathrm{NLL}_{\hat\theta}$. Positive $\Delta$ means the tilt helped.

In [ ]:
EPS = 1e-12
def nll(P, y):
    if P.shape[0] == 0: return float("nan")
    p = np.clip(P[np.arange(P.shape[0]), y.astype(int)], EPS, 1.0)
    return float(-np.log(p).mean())

print(f"{'player':<10} {'split':<7} {'head':<8} {'N':>4} {'NLL_0':>8} {'NLL_θ':>8} {'Δ':>8}")
for (p, sp, head), (P0, Pt, y) in PRED.items():
    n0, nt = nll(P0, y), nll(Pt, y)
    delta = n0 - nt
    print(f"{p:<10} {sp:<7} {head:<8} {y.size:>4} {n0:>8.4f} {nt:>8.4f} {delta:>+8.4f}")

## Metric 2 — Brier score

Same structure as NLL but using `multiclass_brier`. Lower is better; positive $\Delta$ means the tilt helped.

In [ ]:
def mean_brier(P, y):
    if P.shape[0] == 0: return float("nan")
    return float(np.mean([multiclass_brier(P[i], int(y[i])) for i in range(P.shape[0])]))

print(f"{'player':<10} {'split':<7} {'head':<8} {'N':>4} {'Brier_0':>9} {'Brier_θ':>9} {'Δ':>9}")
for (p, sp, head), (P0, Pt, y) in PRED.items():
    b0, bt = mean_brier(P0, y), mean_brier(Pt, y)
    delta = b0 - bt
    print(f"{p:<10} {sp:<7} {head:<8} {y.size:>4} {b0:>9.4f} {bt:>9.4f} {delta:>+9.4f}")

## Metric 3 — Top-1 accuracy and argmax flip rate

* `acc_0`, `acc_θ`: fraction of rows whose argmax matches the realised action.
* **flip rate**: fraction of rows where $\arg\max_a P_0(a\mid\phi) \neq \arg\max_a P_{\hat\theta}(a\mid\phi)$. The tilt is changing the modal *decision*, not just shaving probabilities. A high flip rate with a high $\Delta\,\text{NLL}$ together would be unusually strong evidence that $\hat\theta$ is doing real work.

In [ ]:
def top1(P, y):
    if P.shape[0] == 0: return float("nan")
    return float((P.argmax(axis=1) == y.astype(int)).mean())

def flip_rate(P0, Pt):
    if P0.shape[0] == 0: return float("nan")
    return float((P0.argmax(axis=1) != Pt.argmax(axis=1)).mean())

print(f"{'player':<10} {'split':<7} {'head':<8} {'N':>4} {'acc_0':>7} {'acc_θ':>7} {'flips':>7}")
for (p, sp, head), (P0, Pt, y) in PRED.items():
    print(f"{p:<10} {sp:<7} {head:<8} {y.size:>4} "
          f"{top1(P0, y):>7.3f} {top1(Pt, y):>7.3f} {flip_rate(P0, Pt):>7.3f}")

## Metric 4 — Realised-action shift $|\Delta P(a^\star)|$

For each row, the absolute change in the predicted probability of the action that *actually happened*. This tells us whether the tilt is *aimed* at the right action. Per row we compute $|P_{\hat\theta}(a^\star) - P_0(a^\star)|$ and report the mean and the median, plus a histogram.

In [ ]:
def realised_shifts(P0, Pt, y):
    if P0.shape[0] == 0: return np.zeros(0)
    idx = np.arange(P0.shape[0])
    return np.abs(Pt[idx, y.astype(int)] - P0[idx, y.astype(int)])

print(f"{'player':<10} {'split':<7} {'head':<8} {'N':>4} {'mean|Δ|':>9} {'median|Δ|':>11} {'p95|Δ|':>9}")
shift_arrays = {}
for (p, sp, head), (P0, Pt, y) in PRED.items():
    s = realised_shifts(P0, Pt, y)
    shift_arrays[(p, sp, head)] = s
    if s.size == 0:
        print(f"{p:<10} {sp:<7} {head:<8} {0:>4}")
        continue
    print(f"{p:<10} {sp:<7} {head:<8} {s.size:>4} {s.mean():>9.4f} {np.median(s):>11.4f} {np.percentile(s, 95):>9.4f}")

# histograms — online split only, all heads side by side, per player
fig, axes = plt.subplots(len(PLAYERS), 3, figsize=(13, 3.5 * len(PLAYERS)), sharex=True)
if len(PLAYERS) == 1:
    axes = np.array([axes])
for i, p in enumerate(PLAYERS):
    for j, head in enumerate(("preflop", "facing", "no_bet")):
        ax = axes[i, j]
        s = shift_arrays[(p, "online", head)]
        if s.size == 0:
            ax.set_title(f"{p} {head} (no rows)")
            continue
        ax.hist(s, bins=20, edgecolor="black")
        ax.set_title(f"{p} — {head} (online, n={s.size})")
        ax.set_xlabel("|ΔP(a★)|")
        ax.set_ylabel("rows")
        ax.grid(alpha=0.3)
fig.suptitle("Per-row absolute shift in P(realised action) caused by adding $\\hat\\theta$")
fig.tight_layout()
plt.show()

## Metric 5 — Held-out gradient norm at $\hat\theta$

EM stops when the M-step gradient norm falls below `1e-5` *on the EM-split data*. The held-out analogue answers the generalisation question: at $\hat\theta$, is the M-step gradient on the **online split** also small?

We use the supervised form of the gradient (since we know the realised hand for each row, we don't need the latent posterior $q$ — the per-row contribution becomes $u(a^\star) - \bar u_\theta(h^\star, s)$):

$$g_{\hat\theta}(\text{online}) \;=\; \frac{1}{N}\sum_{i=1}^{N} \Big[u(a^\star_i) - \sum_a P_{\hat\theta}(a \mid h^\star_i, s_i)\,u(a)\Big] \;-\; \lambda\,\hat\theta.$$

(This is the same gradient EM uses, but evaluated on out-of-EM rows with $q$ degenerate at the true hand.) A value substantially larger than the EM tolerance ($10^{-5}$) means $\hat\theta$ would move if EM had access to these rows — it has not generalised.

In [ ]:
L2_PENALTY = 0.25  # same as M_step default in run_preflop_em / run_postflop_theta_em

def utility_vec(action, p_base, n_actions):
    """u_k(a) = 1[a=k] - P_base(k); length n_actions."""
    e = np.zeros(n_actions)
    e[action] = 1.0
    return e - p_base

def heldout_gradient(P0_unused, Pt, y, theta_hat, n_actions):
    """Per-row supervised gradient, averaged, then minus L2 penalty.
    Uses Pt as P_θ (the action distribution under the fitted θ).
    """
    if Pt.shape[0] == 0:
        return np.zeros(n_actions)
    grads = np.zeros(n_actions)
    for i in range(Pt.shape[0]):
        u_obs = utility_vec(int(y[i]), Pt[i], n_actions)
        # expected utility under P_θ with itself as p_base is identically zero, so
        # the per-row contribution simplifies to u_obs (= 1[a=y] - P_θ(a)).
        grads += u_obs
    grads /= Pt.shape[0]
    grads -= L2_PENALTY * np.asarray(theta_hat[:n_actions])
    return grads

print(f"{'player':<10} {'split':<7} {'head':<8} {'||g||':>9}  components (g_fold/g_call/g_raise)")
for (p, sp, head), (P0, Pt, y) in PRED.items():
    if head == "preflop":
        theta_hat = THETA_PRE[p]
        n_actions = 3
    elif head == "facing":
        theta_hat = THETA_POST[p]
        n_actions = 3
    else:
        theta_hat = THETA_POST[p]
        n_actions = 2
    g = heldout_gradient(P0, Pt, y, theta_hat, n_actions)
    gn = float(np.linalg.norm(g))
    comps = np.round(g, 4)
    print(f"{p:<10} {sp:<7} {head:<8} {gn:>9.5f}  {comps}")

## Summary table — online split only

All five metrics on the held-out (online) split, in one place.

In [ ]:
rows_summary = []
for (p, sp, head), (P0, Pt, y) in PRED.items():
    if sp != "online":
        continue
    s = shift_arrays[(p, sp, head)]
    if head == "preflop":
        theta_hat, n_actions = THETA_PRE[p], 3
    elif head == "facing":
        theta_hat, n_actions = THETA_POST[p], 3
    else:
        theta_hat, n_actions = THETA_POST[p], 2
    g = heldout_gradient(P0, Pt, y, theta_hat, n_actions)
    rows_summary.append({
        "player": p, "head": head, "N": int(y.size),
        "ΔNLL":   nll(P0, y) - nll(Pt, y),
        "ΔBrier": mean_brier(P0, y) - mean_brier(Pt, y),
        "acc_θ":  top1(Pt, y),
        "flips":  flip_rate(P0, Pt),
        "mean|Δ|": float(s.mean()) if s.size else float("nan"),
        "||g||":  float(np.linalg.norm(g)),
    })

header = ["player", "head", "N", "ΔNLL", "ΔBrier", "acc_θ", "flips", "mean|Δ|", "||g||"]
print("  ".join(f"{h:<8}" for h in header))
for r in rows_summary:
    cells = [r["player"], r["head"], str(r["N"])]
    for k in ("ΔNLL", "ΔBrier", "acc_θ", "flips", "mean|Δ|", "||g||"):
        cells.append(f"{r[k]:+.4f}" if k.startswith("Δ") else f"{r[k]:.4f}")
    print("  ".join(f"{c:<8}" for c in cells))

## Reading the results

* **$\Delta\mathrm{NLL} > 0$ on the online split** is the headline success criterion. Magnitudes will be small here because $\hat\theta$ itself is small (regularised by L2 against very few bundles).
* **flip rate near zero** with a positive $\Delta\mathrm{NLL}$ means the tilt is working purely through probability calibration — useful for downstream Bayes inversion (range filtering) even though the modal decision never changes.
* **mean $|\Delta P(a^\star)|$** vs **mean $|\Delta P|$ over all actions** distinguishes a well-aimed tilt from a noisy one. If the realised-action shift is comparable to or larger than the average action's shift, the tilt is moving probability mass *toward* the realised action on average.
* **held-out gradient norm**: if it's the same order as the EM tolerance ($10^{-5}$), $\hat\theta$ generalises. If it's $>10^{-2}$ on the online split, EM converged on the EM split but the online data is pulling $\theta$ in a different direction — overfitting or distribution shift.
* If $\Delta\mathrm{NLL}$ is **negative on online but positive on EM**, the tilt is overfit to the EM-split rows. Increase the L2 penalty or reduce the number of EM outer iterations.
* Two players with very different tilt magnitudes (Pluribus's postflop $\theta$ is much larger than Gogo's) will show very different per-row $|\Delta|$. That's expected — the metric just makes the asymmetry quantitative.

These metrics evaluate $\theta$ **conditional on the trained $B$** — they cannot disentangle ``the tilt is bad'' from ``the baseline it tilts is bad.'' If the global-prior evaluation also shows the baseline missing on a head, expect the tilt to look weak there regardless of $\hat\theta$ quality.